# 02.05 · Structured agent output

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lesson 02.03 (`create_agent`). Lesson 01.05 (structured output from a model). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Return** a validated Pydantic object from an agent with `response_format`, and read it from `result["structured_response"]`.
2. **Explain** the two strategies: `ToolStrategy` (an extra "answer" tool) and `ProviderStrategy` (the provider's JSON-schema mode), and what passing a bare schema chooses.
3. **Pick** the right strategy for Groq: `ToolStrategy` whenever the agent has tools.
4. **Handle** validation errors and multiple answers with `handle_errors`.
5. **Offer** alternative answer types with a `Union` (for example, an answer or a refusal).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output) | **Main page.** `response_format`, `ToolStrategy`, `ProviderStrategy`, `handle_errors`, union types |
| [What's new in LangChain v1](https://docs.langchain.com/oss/python/releases/langchain-v1) | Why structured output moved into the main agent loop |
| [Groq: structured outputs](https://console.groq.com/docs/structured-outputs) | Groq's JSON-schema mode and its limits (streaming and tool use) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Why an agent should return an object

An agent's last message is prose: *"You spent about 1.69 million on groceries."* Your app needs `1694000` as a number, the category, maybe a list of caveats. Parsing prose with regexes breaks the first time the wording changes.

`create_agent(..., response_format=Schema)` makes the **last step** of the loop produce a validated `Schema` instance. In v1 this happens **inside** the loop (the old `create_react_agent` made an extra model call after it), so the model can call tools, then answer in the schema, and **retry** if validation fails.

In [ ]:
# 🧪 Offline: the answer schema we'll use, and an agent that returns it.
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import AIMessage
from mastery.agents import format_transcript
from mastery.testing import ScriptedChatModel
from mastery.tools import calculator


class SpendingAnswer(BaseModel):
    """Final answer to a question about the user's spending."""

    answer: str = Field(description="One or two sentences for the user.")
    total_cop: int = Field(ge=0, description="The total in Colombian pesos, copied from a tool result.")
    categories: list[str] = Field(default_factory=list, description="Spending categories the total covers.")


scripted = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "1694000 + 417600"}, "id": "c1"}]),
    AIMessage("", tool_calls=[{"name": "SpendingAnswer", "id": "a1", "args": {
        "answer": "Groceries and restaurants cost 2,111,600 COP in September.",
        "total_cop": 2111600, "categories": ["groceries", "restaurants"]}}]),
])
agent = create_agent(scripted, tools=[calculator], response_format=ToolStrategy(SpendingAnswer))
result = agent.invoke({"messages": [{"role": "user", "content": "Groceries 1,694,000 + restaurants 417,600 in September?"}]})

answer = result["structured_response"]
print(type(answer).__name__, "→", answer)
print("total as an int:", answer.total_cop + 0)
print("tools the model was given:", scripted.bound_tools, "| tool_choice:", scripted.tool_choice)
print()
print(format_transcript(result["messages"]))

**You should now see** a `SpendingAnswer` object with `total_cop=2111600`, the bound tools `['calculator', 'SpendingAnswer']` with `tool_choice: any`, and a transcript that ends with a `SpendingAnswer(...)` call and a tool message *"Returning structured response: ..."*.

### What just happened? (`ToolStrategy`)

- The schema became **one more tool**, named after the class, with the class docstring as its description.
- The agent bound tools with `tool_choice="any"`: the model must call *some* tool every turn, so it can't end with plain text; it ends by "calling" `SpendingAnswer`.
- When that call validates, the agent stores the object in `structured_response`, adds a `ToolMessage` to keep the history valid, and stops.
- Field descriptions are part of the prompt. *"copied from a tool result"* is an instruction.

---
## Part 2 · Two strategies, and which one "auto" picks

| | `ToolStrategy(Schema)` | `ProviderStrategy(Schema)` |
|---|---|---|
| How | Schema becomes a tool; `tool_choice="any"` | Provider's native JSON-schema mode (`response_format`) |
| Works with | any model with tool calling | models/providers that support JSON schema |
| With other tools | ✅ | depends on the provider (**not on Groq**) |
| Retries on validation errors | ✅ `handle_errors` | ❌ (a failed parse raises) |
| Union of schemas | ✅ one tool per schema | ❌ |

If you pass a **bare schema** (`response_format=SpendingAnswer`), LangChain chooses: `ProviderStrategy` when the model's profile says it supports native structured output, otherwise `ToolStrategy`. `ChatGroq` reports `structured_output: True` for GPT-OSS, so **auto picks `ProviderStrategy`**.

Groq's [structured outputs page](https://console.groq.com/docs/structured-outputs) says: *"Streaming and tool use are not currently supported with Structured Outputs."* So an agent with tools + a bare schema sends a request Groq refuses. **Course rule: with Groq, always write `ToolStrategy(...)` explicitly when the agent has tools.** Use `ProviderStrategy` only for tool-less agents, and don't stream it.

In [ ]:
# 🧪 Offline: what ChatGroq's profile says (this is what "auto" reads). No call is made.
from langchain_groq import ChatGroq

profile = ChatGroq(model=settings.model_fast, api_key="gsk_offline_inspection_only").profile or {}
print({k: profile.get(k) for k in ("tool_calling", "structured_output", "max_input_tokens")})
print("auto strategy would be:", "ProviderStrategy" if profile.get("structured_output") else "ToolStrategy")

In [ ]:
# 💸 Live: see the Groq limit for yourself: bare schema + tools (expect an error), then ToolStrategy (works).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    question = {"messages": [{"role": "user", "content": "Use the calculator: what is 1694000 + 417600? Report it as the total."}]}

    try:
        auto = create_agent(llm, tools=[calculator], response_format=SpendingAnswer).invoke(question, {"recursion_limit": 8})
        print("auto (ProviderStrategy) worked:", auto["structured_response"])
    except Exception as exc:  # Groq refuses json_schema + tools; the exact class/text may vary.
        print("auto (ProviderStrategy) failed:", type(exc).__name__, str(exc)[:160])

    explicit = create_agent(llm, tools=[calculator], response_format=ToolStrategy(SpendingAnswer)).invoke(question, {"recursion_limit": 8})
    print("ToolStrategy:", explicit["structured_response"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (as of 2026-10-12) the auto attempt fail with a 400 `BadRequestError` from Groq, and `ToolStrategy` return `total_cop=2111600`. If the first line says *worked*, Groq has added support: note it in your research log, and keep `ToolStrategy` anyway until you've tested it on your model.

In [ ]:
# 💸 Live: ProviderStrategy is fine for a tool-less extraction agent (1 call, no streaming).
if HAVE_GROQ_KEY:
    from langchain.agents.structured_output import ProviderStrategy

    extractor = create_agent(llm, tools=[], response_format=ProviderStrategy(SpendingAnswer),
                             system_prompt="Extract the spending total the user states. Do not compute anything new.")
    extracted = extractor.invoke({"messages": [{"role": "user", "content": "My rent was 1,850,000 COP this month (housing)."}]})
    print(extracted["structured_response"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

---
## Part 3 · When the answer doesn't validate: `handle_errors`

The model may produce an answer that breaks the schema (a negative total, a missing field) or call the answer tool **twice**. With `ToolStrategy`, `handle_errors` decides what happens:

| `handle_errors=` | Behaviour |
|---|---|
| `True` (default) | Send the error back as a `ToolMessage` and let the model try again |
| a `str` | Same, with your message instead of the error text |
| an exception type / tuple | Retry only for those; raise others |
| a callable `f(exc) -> str` | Build the retry message yourself |
| `False` | Raise `StructuredOutputValidationError` / `MultipleStructuredOutputsError` |

In [ ]:
# 🧪 Offline: a negative total fails validation (ge=0); the model fixes it on the second try.
fixing = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "SpendingAnswer", "id": "v1", "args": {"answer": "Refund month!", "total_cop": -50000}}]),
    AIMessage("", tool_calls=[{"name": "SpendingAnswer", "id": "v2", "args": {"answer": "Net spend was 0 COP.", "total_cop": 0}}]),
])
out = create_agent(fixing, tools=[calculator], response_format=ToolStrategy(SpendingAnswer)).invoke(
    {"messages": [{"role": "user", "content": "Net spend?"}]})
print(format_transcript(out["messages"]))
print("\nstructured_response:", out["structured_response"])

**You should now see** the first answer rejected with *"Error: Failed to parse structured output for tool 'SpendingAnswer' ... greater than or equal to 0"*, then the corrected answer accepted. Two model calls, no exception.

In [ ]:
# 🧪 Offline: two answers in one turn, and handle_errors=False.
from langchain.agents.structured_output import StructuredOutputValidationError

double = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[
        {"name": "SpendingAnswer", "id": "d1", "args": {"answer": "A", "total_cop": 1}},
        {"name": "SpendingAnswer", "id": "d2", "args": {"answer": "B", "total_cop": 2}},
    ]),
    AIMessage("", tool_calls=[{"name": "SpendingAnswer", "id": "d3", "args": {"answer": "B", "total_cop": 2}}]),
])
out = create_agent(double, tools=[calculator], response_format=ToolStrategy(SpendingAnswer)).invoke(
    {"messages": [{"role": "user", "content": "Total?"}]})
print("after retry:", out["structured_response"])

strict = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "SpendingAnswer", "id": "s1", "args": {"answer": "?", "total_cop": -1}}])])
try:
    create_agent(strict, tools=[calculator], response_format=ToolStrategy(SpendingAnswer, handle_errors=False)).invoke(
        {"messages": [{"role": "user", "content": "Total?"}]})
except StructuredOutputValidationError as exc:
    print("handle_errors=False →", type(exc).__name__)

**You should now see** `after retry: answer='B' total_cop=2` (the agent rejected both calls with *"Model incorrectly returned multiple structured responses"* and the model answered once), then `handle_errors=False → StructuredOutputValidationError`. Use `False` in tests, where you want failures to be loud.

---
## Part 4 · Several answer types: a `Union`

Sometimes "the answer" can have different shapes. The most useful case: **an answer or a refusal**. With `ToolStrategy(SpendingAnswer | OutOfScope)`, each schema becomes a separate tool and the model picks one. Your code branches on the type, not on wording.

`tool_message_content="..."` sets the text of the final `ToolMessage` (default: *"Returning structured response: ..."*), which is useful if you show the history to users.

In [ ]:
# 🧪 Offline: answer-or-refuse, decided by the type of structured_response.
class OutOfScope(BaseModel):
    """Use when the question is not about the user's own spending data."""

    reason: str = Field(description="Short explanation for the user of why you can't answer.")


refusing = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "OutOfScope", "id": "o1", "args": {"reason": "I only know your expense records, not stock prices."}}]),
])
union_agent = create_agent(refusing, tools=[calculator],
                           response_format=ToolStrategy(SpendingAnswer | OutOfScope, tool_message_content="Answer recorded."))
out = union_agent.invoke({"messages": [{"role": "user", "content": "Should I buy Ecopetrol shares?"}]})

response = out["structured_response"]
print("tools offered:", refusing.bound_tools)
match response:
    case OutOfScope(reason=reason):
        print("REFUSED:", reason)
    case SpendingAnswer(total_cop=total):
        print("ANSWER :", total)
print("last ToolMessage:", out["messages"][-1].content)

In [ ]:
# 💸 Live: one in-scope and one out-of-scope question (2–4 short calls).
if HAVE_GROQ_KEY:
    scoped = create_agent(
        llm, tools=[calculator], response_format=ToolStrategy(SpendingAnswer | OutOfScope),
        system_prompt=("You answer questions about the user's spending using the calculator for all arithmetic. "
                       "If a question is not about their spending, answer with OutOfScope."))
    for q in ("I spent 1,850,000 on rent and 264,900 on utilities. Total?", "What's the weather in Bogotá?"):
        r = scoped.invoke({"messages": [{"role": "user", "content": q}]}, {"recursion_limit": 8})
        print(f"{q[:45]:<46} → {r['structured_response']!r}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** a `SpendingAnswer(... total_cop=2114900 ...)` for the first question and an `OutOfScope(reason=...)` for the weather. Project 02 uses exactly this pattern for its refusal requirement.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| 400 from Groq on the first call of an agent with tools | Bare schema → auto picks `ProviderStrategy`; Groq doesn't support JSON schema + tools | `response_format=ToolStrategy(Schema)` |
| `KeyError: 'structured_response'` | No `response_format`, or the run stopped early (limit, error) | Check `"structured_response" in result` and the last messages |
| Answer has plausible but wrong numbers | Model wrote the number itself | Field description *"copied from a tool result"*; test that numbers appear in `ToolMessage`s (Project 02) |
| `StructuredOutputValidationError` | `handle_errors=False`, or a `ProviderStrategy` parse failure | Use `ToolStrategy` (retries) or catch it |
| The model refuses with text instead of `OutOfScope` | It can't end with text under `ToolStrategy`; if it does, the prompt conflicts | Describe when to use each schema in its docstring and the system prompt |
| Pydantic `Field` constraints ignored by the model | Constraints are hints in the JSON schema | Keep them: validation + retry enforce them anyway |

## ⚠️ Production notes

> ⚠️ **Production note: with Groq, write `ToolStrategy` explicitly.**
> A bare schema lets LangChain pick `ProviderStrategy` from the model profile. On Groq that conflicts with tools (and streaming). Being explicit also protects you when a profile changes after an upgrade. Verified against the docs and the local Groq stand-in on 2026-10-12; confirm with the 💸 cell on your account.

> ⚠️ **Production note: structured output is an API contract.**
> Version your answer schemas (`SpendingAnswerV1`), keep fields optional where the model may not know, and never let the UI depend on wording in `answer`. Log validation retries: a rising retry rate is an early sign of a prompt or model regression.

> ⚠️ **Production note: refusals as data.**
> A `Union` with an `OutOfScope` type makes refusals measurable (count them, alert on spikes) and testable, instead of guessing from text.

## 🏋️ Exercises

1. **(Easy)** Add `currency: Literal["COP", "USD"] = "COP"` to `SpendingAnswer` and rerun Part 1. What does the schema tool's description look like now? (`convert_to_openai_tool(SpendingAnswer)`)
2. **(Medium)** Use `handle_errors="Your answer must have total_cop >= 0 and copy numbers from tool results."` and script a failing first answer. Check the `ToolMessage` text.
3. **(Medium)** Add a third type, `NeedMoreInfo(question: str)`, to the union, for questions like *"How much did I spend?"* (no month). Script it and branch on it with `match`.
4. **(Stretch)** Write a check `numbers_from_tools(result) -> bool` that returns `True` only if `structured_response.total_cop` appears (as digits) in some `ToolMessage` of `result["messages"]`. Run it on Part 1's result and on a scripted answer that invents a number.

<details>
<summary><b>Show solutions</b></summary>

**4.**
```python
import re
from langchain.messages import ToolMessage

def numbers_from_tools(result) -> bool:
    target = str(result["structured_response"].total_cop)
    tool_text = " ".join(m.content for m in result["messages"]
                         if isinstance(m, ToolMessage) and m.name != "SpendingAnswer")
    return target in re.sub(r"[,.](?=\d{3})", "", tool_text)

print(numbers_from_tools(result))   # True: 2111600 came from the calculator
```
Project 02 uses this idea for its "no arithmetic by the LLM" acceptance criterion.
</details>

## ✅ Checkpoint

- [ ] I can build an agent with `response_format=ToolStrategy(Schema)` and read `structured_response`.
- [ ] I can explain how `ToolStrategy` works (an extra tool + `tool_choice="any"`).
- [ ] I know what a bare schema chooses and why that fails with Groq + tools.
- [ ] I can configure `handle_errors` and know what it does with validation errors and multiple answers.
- [ ] I can return answer-or-refusal with a `Union` and branch on the type.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read the `ProviderStrategy` section of the [structured output docs](https://docs.langchain.com/oss/python/langchain/structured-output) and the `strict` parameter. Then read Groq's [structured outputs](https://console.groq.com/docs/structured-outputs) page: which models support strict mode, and is tool use still listed as unsupported? Record the date and the answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `response_format` turns the agent's final step into a validated object in `structured_response`.
- `ToolStrategy` = schema as an extra tool + forced tool use; it retries on errors and supports unions.
- A bare schema picks `ProviderStrategy` when the profile allows; on Groq with tools, that fails, so be explicit.
- `handle_errors` controls retries; `False` is for tests.
- Answer-or-refusal unions make scope limits explicit and testable.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 02.06 · Streaming agents** → [`02-06-streaming-agents.ipynb`](02-06-streaming-agents.ipynb).

[Module 02 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)